# BERTimbau — classificação multirrótulo de proposições por tema (Fase 4)

Aqui treinamos o **modelo avançado**: o *fine-tuning* do **BERTimbau** para dizer os **temas**
de cada proposição a partir da **ementa**.

**Meta:** superar a régua do baseline (macro-F1 **0,639** no corte ≥200, com limiar ajustado),
medindo no **mesmo teste** (reusamos `particao_treino_val_teste.csv`).

> ⚠️ **Pré-requisito:** rode antes o `02_baseline_tfidf.ipynb`. Ele gera dois arquivos que
> este notebook usa: `particao_treino_val_teste.csv` (a divisão, lida no Passo 1) e
> `resultados_baseline.csv` (a comparação do Passo 8).

**Precisa de GPU NVIDIA.** O *fine-tuning* foi treinado numa **RTX 3060 (12 GB)** — fica de
referência: uma GPU modesta já dá conta. Na CPU o treino é inviável de tão lento; sem GPU,
rode no Colab. Instale:
```
pip install torch transformers accelerate scikit-learn pandas
```
> Notas:
> - O `accelerate` é **obrigatório** para o `Trainer` (sem ele dá `ImportError`).
> - No Windows, `pip install torch` traz a versão **só-CPU**. Para usar a GPU, instale a
>   build com CUDA — ex.: `pip install torch --index-url https://download.pytorch.org/whl/cu124`
>   (troque `cu124` pela sua versão de CUDA).
> - Este notebook usa a API do **`transformers` 5.x** (ex.: `eval_strategy`); em versões 4.x
>   o parâmetro se chamava `evaluation_strategy`.
> - Se instalar pacotes com o notebook aberto, **reinicie o kernel** depois.

## Passo 0 — Conferir a GPU
Tem que aparecer `GPU disponível: True`. Se aparecer `False`, reinstale o torch CUDA (acima)
ou rode no Colab.

In [ ]:
import os
os.makedirs("dados", exist_ok=True)
import torch
print("GPU disponivel:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Placa:", torch.cuda.get_device_name(0))

## Passo 1 — Carregar dados e a divisão treino/validação/teste
Reusamos a divisão da Fase 3 (mesmo teste). Rótulos viram multi-hot `Y` (32 colunas).

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MultiLabelBinarizer

dados = pd.read_csv("dados/proposicoes_temas.csv")
dados["ementa"] = dados["ementa"].fillna("").astype(str)
particao = pd.read_csv("dados/particao_treino_val_teste.csv")
dados = dados.merge(particao, on="id")

mlb = MultiLabelBinarizer()
Y = mlb.fit_transform(dados["temas"].str.split("|"))
nomes_temas = list(mlb.classes_)
mask_sup200 = Y.sum(axis=0) >= 200

def separar(parte):
    m = (dados["particao"] == parte).values
    return dados.loc[m, "ementa"].tolist(), Y[m]

textos_tr, Y_tr = separar("treino")
textos_val, Y_val = separar("validacao")
textos_te, Y_te = separar("teste")
print(f"treino: {len(textos_tr)} | validacao: {len(textos_val)} | teste: {len(textos_te)}")

## Passo 1b — Diagnóstico: quantos *tokens* têm as ementas?
O BERT lê em tokens (subpalavras), mais que palavras. Medimos para escolher o `max_length`
com base em dados. (Já medido: mediana 45, p99=135 → escolhemos 192, trunca ~0,16%.)

In [ ]:
from transformers import AutoTokenizer

MODELO = "neuralmind/bert-base-portuguese-cased"
tokenizer = AutoTokenizer.from_pretrained(MODELO)

tamanhos = pd.Series([len(x) for x in tokenizer(dados["ementa"].tolist(),
                     add_special_tokens=True, truncation=False)["input_ids"]])
print(tamanhos.describe(percentiles=[.5, .9, .95, .99]).round(1).to_string())
for L in [128, 192, 256]:
    print(f"  max_length={L}: {100*(tamanhos > L).mean():.2f}% truncadas")

## Passo 2 — Preparar os dados no formato do BERT (`max_length=192`)

In [ ]:
MAX_LENGTH = 192

class DatasetEmentas(torch.utils.data.Dataset):
    def __init__(self, textos, rotulos):
        self.enc = tokenizer(textos, truncation=True, padding="max_length", max_length=MAX_LENGTH)
        self.rotulos = rotulos.astype("float32")
    def __len__(self):
        return len(self.rotulos)
    def __getitem__(self, i):
        item = {k: torch.tensor(v[i]) for k, v in self.enc.items()}
        item["labels"] = torch.tensor(self.rotulos[i])
        return item

ds_tr  = DatasetEmentas(textos_tr,  Y_tr)
ds_val = DatasetEmentas(textos_val, Y_val)
ds_te  = DatasetEmentas(textos_te,  Y_te)
print("Datasets prontos.")

## Passo 3 — Preparar o carregador do modelo
O modelo terá **32 saídas** e `problem_type="multi_label_classification"` (sigmoide + BCE).
Como vamos testar vários learning_rates, o modelo é **criado do zero dentro do sweep**
(Passo 5), um para cada learning_rate. Aqui só deixamos a classe importada.

In [ ]:
from transformers import AutoModelForSequenceClassification
print("Pronto para criar o modelo:", MODELO, "->", len(nomes_temas), "saidas (multirrotulo).")

## Passo 4 — Como medir o desempenho (durante o treino)
Depois de cada época, na validação: macro-F1 (32 e ≥200) e micro-F1, no limiar fixo 0,50.

In [ ]:
from sklearn.metrics import f1_score

def sigmoide(x):
    return 1 / (1 + np.exp(-x))

def calcular_metricas(predicao):
    logits = predicao.predictions
    if isinstance(logits, tuple):
        logits = logits[0]
    y_pred = (sigmoide(logits) >= 0.5).astype(int)
    y_true = predicao.label_ids.astype(int)
    return {
        "macro_f1_32":     f1_score(y_true, y_pred, average="macro", zero_division=0),
        "macro_f1_sup200": f1_score(y_true[:, mask_sup200], y_pred[:, mask_sup200],
                                    average="macro", zero_division=0),
        "micro_f1":        f1_score(y_true, y_pred, average="micro", zero_division=0),
    }

## Passo 5 — Buscar o melhor learning_rate (varredura enxuta)

O `learning_rate` é o hiperparâmetro que mais afeta o fine-tuning de BERT. Testamos 3 valores
da faixa recomendada (**2e-5, 3e-5, 5e-5**) e ficamos com o melhor **na validação**
(`macro_f1_sup200`). Cada treino usa: `warmup_ratio=0.1` (aquece o LR no início, dá
estabilidade), `weight_decay=0.01`, `batch=16` e até **6 épocas com early stopping**
(ele para sozinho na melhor época — então não precisamos varrer "épocas").

> São ~3 treinos. Se faltar memória, baixe `per_device_train_batch_size` para 8.
> A escolha é feita só na validação; o teste fica intocado até o fim.

In [ ]:
from transformers import TrainingArguments, Trainer, EarlyStoppingCallback
import gc

def treinar(lr):
    """Treina um BERTimbau do zero com um dado learning_rate; devolve (trainer, score_val)."""
    modelo = AutoModelForSequenceClassification.from_pretrained(
        MODELO, num_labels=len(nomes_temas),
        problem_type="multi_label_classification")
    args = TrainingArguments(
        output_dir=f"bert_saida/lr_{lr:.0e}",
        num_train_epochs=6,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        learning_rate=lr,
        weight_decay=0.01,
        warmup_ratio=0.1,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="macro_f1_sup200",
        greater_is_better=True,
        fp16=torch.cuda.is_available(),
        logging_steps=50,
        report_to="none",
        save_total_limit=1,
    )
    trainer = Trainer(
        model=modelo, args=args,
        train_dataset=ds_tr, eval_dataset=ds_val,
        compute_metrics=calcular_metricas,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
    )
    trainer.train()
    score = trainer.evaluate()["eval_macro_f1_sup200"]
    return trainer, score

melhor_trainer, melhor_lr, melhor_score, historico = None, None, -1.0, []
for lr in [2e-5, 3e-5, 5e-5]:
    print(f"\n===== Treinando com learning_rate = {lr:.0e} =====")
    tr, score = treinar(lr)
    historico.append((lr, score))
    print(f"lr={lr:.0e}  ->  val macro_f1_sup200 = {score:.3f}")
    if score > melhor_score:
        melhor_score, melhor_lr, melhor_trainer = score, lr, tr
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("\nResumo da busca (validacao):")
for lr, s in historico:
    print(f"  lr={lr:.0e}: {s:.3f}" + ("   <-- melhor" if lr == melhor_lr else ""))

trainer = melhor_trainer   # usa o melhor modelo daqui pra frente
print(f"\nMelhor learning_rate: {melhor_lr:.0e}")

## Passo 6 — Ajustar o limiar de cada tema (na validação)

Igual ao baseline: em vez do 0,50 fixo, escolhemos **para cada tema** o limiar que maximiza
o F1 daquele tema **na validação**, e aplicamos no teste. Mesma técnica nos dois modelos →
comparação justa.

In [ ]:
def melhores_limiares(y_val, prob_val, grade=np.arange(0.05, 0.96, 0.05)):
    ts = []
    for j in range(y_val.shape[1]):
        f1s = [f1_score(y_val[:, j], (prob_val[:, j] >= t).astype(int), zero_division=0)
               for t in grade]
        ts.append(grade[int(np.argmax(f1s))] if max(f1s) > 0 else 0.5)
    return np.array(ts)

def probabilidades(ds):
    pred = trainer.predict(ds)
    logits = pred.predictions[0] if isinstance(pred.predictions, tuple) else pred.predictions
    return sigmoide(logits)

prob_val   = probabilidades(ds_val)
prob_teste = probabilidades(ds_te)
limiares = melhores_limiares(Y_val.astype(int), prob_val)
print("Limiares ajustados (amostra):")
for nome, t in sorted(zip(nomes_temas, limiares), key=lambda x: x[1])[:6]:
    print(f"  {nome:<40} {t:.2f}")

## Passo 7 — Avaliar no teste: 0,50 vs limiar ajustado

In [ ]:
y_true = Y_te.astype(int)

def resumo(y_pred):
    return (f1_score(y_true, y_pred, average="macro", zero_division=0),
            f1_score(y_true[:, mask_sup200], y_pred[:, mask_sup200], average="macro", zero_division=0),
            f1_score(y_true, y_pred, average="micro", zero_division=0))

pred_05 = (prob_teste >= 0.5).astype(int)
pred_aj = (prob_teste >= limiares).astype(int)
m05, maj = resumo(pred_05), resumo(pred_aj)
print("                         macro(32)  macro(>=200)  micro")
print(f"limiar 0,50:             {m05[0]:.3f}      {m05[1]:.3f}       {m05[2]:.3f}")
print(f"limiar ajustado/tema:    {maj[0]:.3f}      {maj[1]:.3f}       {maj[2]:.3f}")

## Passo 8 — F1 por tema e tabela comparativa com o baseline

In [ ]:
from sklearn.metrics import precision_recall_fscore_support

p, r, f1v, sup = precision_recall_fscore_support(y_true, pred_aj, zero_division=0)
resultados = pd.DataFrame({
    "tema": nomes_temas, "limiar": limiares,
    "precisao": p, "revocacao": r, "f1": f1v, "suporte_teste": sup,
}).sort_values("f1", ascending=False)
resultados.to_csv("dados/resultados_bertimbau.csv", index=False, encoding="utf-8")
print("Salvo: dados/resultados_bertimbau.csv\n")

base = pd.read_csv("dados/resultados_baseline.csv")[["tema", "f1"]].rename(columns={"f1": "f1_baseline"})
comp = (resultados[["tema", "f1"]].rename(columns={"f1": "f1_bertimbau"}).merge(base, on="tema"))
comp["ganho"] = (comp["f1_bertimbau"] - comp["f1_baseline"]).round(3)
print("Ganho do BERTimbau sobre o baseline (ambos com limiar ajustado):")
comp.sort_values("ganho", ascending=False)

## Passo 9 — Salvar o modelo treinado
Para reusar no 2º experimento (discursos) sem treinar de novo.

In [ ]:
trainer.save_model("modelo_bertimbau")
tokenizer.save_pretrained("modelo_bertimbau")
print("Modelo salvo em modelo_bertimbau/")